# EDA — Dataset Sintético de NER Financiero

**Proyecto Final Módulo 2 · NER Financiero con BETO + LoRA · ITAM**

Este notebook explora el dataset generado en `scripts/generate_dataset.py`:
- Balance de clases por entidad (BIO)
- Distribución de longitud de secuencias
- Validación de muestras representativas
- Riesgo de overfitting dado el tamaño del dataset

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / 'src'))

import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from financial_ner.data.schema import validate_dataset, label_distribution

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

with open('../data/raw/financial_ner_dataset.json', encoding='utf-8') as f:
    dataset = json.load(f)

print(f'Dataset cargado: {len(dataset)} ejemplos')
validate_dataset(dataset)
print('✓ Esquema BIO válido')

## 1. Balance de clases (esquema BIO)

In [ ]:
dist = label_distribution(dataset)
dist_sorted = dict(sorted(dist.items(), key=lambda x: -x[1]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1a. Distribución completa (incluye O, que domina)
ax = axes[0]
labels = list(dist_sorted.keys())
counts = list(dist_sorted.values())
colors = ['#95A5A6' if l == 'O' else '#1a2744' for l in labels]
ax.barh(labels, counts, color=colors)
ax.set_title('Distribución de etiquetas BIO\n(incluye O — contexto no-entidad)', fontweight='bold')
ax.set_xlabel('Conteo de tokens')
for i, v in enumerate(counts):
    ax.text(v + 10, i, str(v), va='center', fontsize=9)

# 1b. Solo entidades (sin O) agrupadas por tipo
ax2 = axes[1]
entity_types = ['INSTR', 'MONTO', 'PLAZO', 'TASA', 'ENTIDAD']
entity_totals = {
    e: dist_sorted.get(f'B-{e}', 0) for e in entity_types
}
colors2 = sns.color_palette('husl', len(entity_types))
bars = ax2.bar(entity_totals.keys(), entity_totals.values(), color=colors2)
ax2.set_title('Entidades totales por tipo\n(conteo de spans B-X)', fontweight='bold')
ax2.set_ylabel('Número de entidades')
ax2.bar_label(bars, padding=3)

plt.suptitle('Balance de clases — Dataset NER Financiero', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nNota: MONTO tiene menos ejemplos que las demás categorías.')
print('Esto se documenta como limitación en el reporte final.')

## 2. Distribución de longitud de secuencias

In [ ]:
lengths = [len(ex['tokens']) for ex in dataset]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=20, color='#1a2744', edgecolor='white')
ax.axvline(sum(lengths)/len(lengths), color='#e67e22', linestyle='--',
           label=f'Promedio: {sum(lengths)/len(lengths):.1f} tokens')
ax.set_xlabel('Número de tokens por oración')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de longitud de secuencias', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Min: {min(lengths)} | Max: {max(lengths)} | Promedio: {sum(lengths)/len(lengths):.1f}')
print(f'\nmax_seq_len=64 (configurado) cubre el 100% de las secuencias sin truncar.')

## 3. Ejemplos representativos por categoría

In [ ]:
def find_example_with_entity(dataset, entity_type):
    for ex in dataset:
        if f'B-{entity_type}' in ex['ner_tags']:
            return ex
    return None

print('Un ejemplo por cada tipo de entidad:\n')
for entity in ['INSTR', 'MONTO', 'PLAZO', 'TASA', 'ENTIDAD']:
    ex = find_example_with_entity(dataset, entity)
    print(f'--- {entity} ---')
    print('  Texto:', ' '.join(ex['tokens']))
    print('  Tags: ', ' '.join(ex['ner_tags']))
    print()

## 4. Vocabulario único por categoría — riesgo de overfitting

Con un dataset sintético generado por plantillas, el riesgo principal de overfitting
no es el tamaño total sino la **diversidad de vocabulario real** dentro de cada categoría.
Si el vocabulario es muy pequeño, LoRA puede memorizar valores específicos en lugar de
aprender el patrón de la entidad.

In [ ]:
def extract_entity_spans(dataset, entity_type):
    """Extrae todos los spans de texto de un tipo de entidad."""
    spans = []
    for ex in dataset:
        tokens, tags = ex['tokens'], ex['ner_tags']
        current = []
        for tok, tag in zip(tokens, tags):
            if tag == f'B-{entity_type}':
                if current:
                    spans.append(' '.join(current))
                current = [tok]
            elif tag == f'I-{entity_type}' and current:
                current.append(tok)
            else:
                if current:
                    spans.append(' '.join(current))
                    current = []
        if current:
            spans.append(' '.join(current))
    return spans

print('Vocabulario único por categoría:\n')
for entity in ['INSTR', 'MONTO', 'PLAZO', 'TASA', 'ENTIDAD']:
    spans = extract_entity_spans(dataset, entity)
    unique = set(spans)
    print(f'{entity:10s}: {len(spans):3d} ocurrencias, {len(unique):2d} valores únicos')

print('\nConclusión: el vocabulario diverso (MX + internacional) reduce el riesgo')
print('de que LoRA memorice ejemplos específicos en lugar del patrón BIO general.')